# 🚀 Jina Reranker v2 INT8 ONNX Quantization & Hugging Face Upload

This notebook creates an **INT8 Quantized ONNX** version of `jinaai/jina-reranker-v2-base-multilingual`.

### 📜 License Notice:
The underlying model `jina-reranker-v2-base-multilingual` is licensed under **CC-BY-NC-4.0** (Creative Commons Attribution-NonCommercial 4.0 International).
Use is strictly for non-commercial educational and research purposes. Uploads to Hugging Face will include the `cc-by-nc-4.0` license tag and full attribution to Jina AI GmbH.

### 🎯 Results:
- **Size:** Shrinks from **1.15 GB** to **~280 MB**.
- **Speed:** ~**150 ms – 300 ms** on CPU.
- **Compatibility:** Works directly with ONNX Runtime in `Study-Buddy`.

## 1. Install Dependencies
We use `fastembed` to download the pre-optimized base ONNX graph (avoiding generic `optimum-cli` export bugs), and `onnxruntime` for INT8 quantization.

In [1]:
!pip install -q fastembed onnx onnxruntime huggingface_hub transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 80.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 79.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.1/324.1 kB 34.4 MB/s eta 0:00:00


## 2. Obtain the Base ONNX Model
We initialize the cross-encoder with a local cache directory to get the verified ONNX model and tokenizer files.

In [2]:
import os
from pathlib import Path
from fastembed.rerank.cross_encoder import TextCrossEncoder

MODEL_NAME = "jinaai/jina-reranker-v2-base-multilingual"
BASE_CACHE_DIR = Path("./fastembed_cache").resolve()
BASE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"[1/4] Fetching verified ONNX model for '{MODEL_NAME}'...")
reranker = TextCrossEncoder(model_name=MODEL_NAME, cache_dir=str(BASE_CACHE_DIR))

# Find the downloaded model directory
model_dirs = list(BASE_CACHE_DIR.glob("**/*model*.onnx"))
assert len(model_dirs) > 0, "❌ ONNX model file not found in cache!"
source_model_dir = model_dirs[0].parent
source_onnx_file = model_dirs[0]

print(f"✅ ONNX model retrieved successfully!")
print(f"   Directory: {source_model_dir}")
print(f"   File     : {source_onnx_file.name} ({source_onnx_file.stat().st_size / 1e6:.1f} MB)")

[1/4] Fetching verified ONNX model for 'jinaai/jina-reranker-v2-base-multilingual'...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

✅ ONNX model retrieved successfully!
   Directory: /content/fastembed_cache/models--jinaai--jina-reranker-v2-base-multilingual/snapshots/9cfeff2df7d40d1b78e75e5e9cebec92a99813c9/onnx
   File     : model.onnx (1114.0 MB)


## 3. Quantize to INT8 (Dynamic Quantization)
We quantize the weights to `QInt8`, saving ~75% disk space and accelerating CPU matrix multiplication.

In [3]:
import shutil
from onnxruntime.quantization import quantize_dynamic, QuantType

INT8_EXPORT_DIR = Path("./jina_reranker_int8_onnx").resolve()
INT8_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

output_quantized_onnx = INT8_EXPORT_DIR / "model_quantized.onnx"

print(f"[2/4] Quantizing {source_onnx_file.name} to INT8 dynamic...")
quantize_dynamic(
    model_input=str(source_onnx_file),
    model_output=str(output_quantized_onnx),
    weight_type=QuantType.QInt8,
)

# Copy tokenizer, vocabulary, and config files
for item in source_model_dir.glob("*"):
    if item.name != source_onnx_file.name:
        dest = INT8_EXPORT_DIR / item.name
        if item.is_file():
            shutil.copy2(item, dest)
        elif item.is_dir() and not dest.exists():
            shutil.copytree(item, dest)

orig_mb = source_onnx_file.stat().st_size / 1e6
quant_mb = output_quantized_onnx.stat().st_size / 1e6
print(f"✅ INT8 Quantization Complete!")
print(f"   - Original size : {orig_mb:.1f} MB")
print(f"   - Quantized size: {quant_mb:.1f} MB ({100 - (quant_mb / orig_mb * 100):.1f}% reduction)")

[2/4] Quantizing model.onnx to INT8 dynamic...


✅ INT8 Quantization Complete!
   - Original size : 1114.0 MB
   - Quantized size: 279.2 MB (74.9% reduction)


## 4. Test & Benchmark on CPU
Verify the INT8 model runs in ONNX Runtime and measure per-query latency.

In [4]:
import time
import numpy as np
import onnxruntime as ort
from transformers import AutoTokenizer

print("[3/4] Testing INT8 ONNX Model on CPU...")
tokenizer = AutoTokenizer.from_pretrained(str(INT8_EXPORT_DIR))

opts = ort.SessionOptions()
opts.intra_op_num_threads = os.cpu_count() or 4
session = ort.InferenceSession(str(output_quantized_onnx), sess_options=opts, providers=["CPUExecutionProvider"])

query = "In Distributional RL, why is the Bellman operator a contraction mapping under Wasserstein distance?"
documents = [
    "The Wasserstein metric guarantees a gamma-contraction for the distributional Bellman operator unlike KL divergence.",
    "Paris is the capital of France and features the Eiffel Tower.",
    "Quantile Regression DQN estimates fixed quantile fractions instead of uniform categorical bins.",
    "Photosynthesis is the biological process used by plants to convert light into energy."
]
pairs = [[query, doc] for doc in documents]

inputs = tokenizer(pairs, padding=True, truncation=True, max_length=512, return_tensors="np")
onnx_inputs = {
    "input_ids": inputs["input_ids"],
    "attention_mask": inputs["attention_mask"],
}

# Measure inference latency
start = time.perf_counter()
outputs = session.run(None, onnx_inputs)
latency_ms = (time.perf_counter() - start) * 1000

scores = outputs[0].flatten().tolist()
print(f"⏱️ INT8 ONNX CPU Latency: {latency_ms:.1f} ms for {len(pairs)} candidate pairs!")
print("\nRanked candidate chunks:")
ranked = sorted(zip(scores, documents), key=lambda x: x[0], reverse=True)
for idx, (score, doc) in enumerate(ranked, 1):
    print(f"  {idx}. [Score: {score:.4f}] {doc}")

[3/4] Testing INT8 ONNX Model on CPU...


ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.

## 5. Upload to Hugging Face Hub
Upload the quantized model to your Hugging Face Hub account with the `cc-by-nc-4.0` license.

In [ ]:
from huggingface_hub import HfApi, login

# Step 1: Login (enter your Hugging Face Access Token with WRITE permissions)
login()

# Step 2: Configure your target repository
HF_USERNAME = "your-username"  # <-- ✏️ REPLACE with your HF username
REPO_NAME = "jina-reranker-v2-base-multilingual-int8-onnx"
REPO_ID = f"{HF_USERNAME}/{REPO_NAME}"

# Step 3: Generate Model Card with CC-BY-NC-4.0 License
readme_content = f"""---
license: cc-by-nc-4.0
language:
- multilingual
- en
- ar
tags:
- reranker
- onnx
- int8
- cross-encoder
pipeline_tag: text-classification
base_model: jinaai/jina-reranker-v2-base-multilingual
---

# Jina Reranker v2 Base Multilingual (INT8 ONNX)

This is the ONNX INT8 dynamic quantized version of [jinaai/jina-reranker-v2-base-multilingual](https://huggingface.co/jinaai/jina-reranker-v2-base-multilingual).

### Model Specifications:
- **Format:** ONNX INT8 (`model_quantized.onnx`)
- **Size:** ~280 MB (reduced from ~1.15 GB)
- **CPU Latency:** ~150 ms – 300 ms
- **Languages:** 100+ languages supported including English and Arabic.
- **License:** [CC-BY-NC-4.0](https://creativecommons.org/licenses/by-nc/4.0/) (Non-Commercial Educational & Research Use)
- **Attribution:** Original model developed by Jina AI GmbH (https://jina.ai).

### Use with Study-Buddy:
Add to your `config.env`:
```env
RERANKER_QUANTIZED_MODEL={REPO_ID}
```
"""

with open(INT8_EXPORT_DIR / "README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

# Step 4: Create repo and upload folder
api = HfApi()
api.create_repo(repo_id=REPO_ID, repo_type="model", exist_ok=True)
print(f"[4/4] Uploading files to https://huggingface.co/{REPO_ID} ...")

api.upload_folder(
    folder_path=str(INT8_EXPORT_DIR),
    repo_id=REPO_ID,
    repo_type="model",
)

print(f"\n🎉 Upload complete!")
print(f"Your model is live at: https://huggingface.co/{REPO_ID}")
print(f"\n👉 To use it in Study-Buddy, set in config.env:")
print(f"RERANKER_QUANTIZED_MODEL={REPO_ID}")

[4/4] Uploading files to https://huggingface.co/Hamdy005/jina-reranker-v2-base-multilingual-int8-onnx ...

🎉 Upload complete!
Your model is live at: https://huggingface.co/Hamdy005/jina-reranker-v2-base-multilingual-int8-onnx

👉 To use it in Study-Buddy, set in config.env:
RERANKER_QUANTIZED_MODEL=Hamdy005/jina-reranker-v2-base-multilingual-int8-onnx
